# Stack adversarial-query reproduction — overnight run

Designed for **Runtime → Run all** on an A100 Colab runtime. It restores the database/models, verifies the serving stack, validates vLLM grammar enforcement, runs one bounded medium experiment, and writes a durable morning report to Google Drive.

Default worst-case database time is about 40 minutes (20 initialization points + 20 BO calls, each involving two queries capped at 30 seconds). All experiment logs are written directly to Drive.

In [ ]:
# Edit only this cell when changing the overnight budget.
from datetime import datetime
from pathlib import Path
import os

BRANCH = "codex/stack-cpu-repro"
REPO_URL = "https://github.com/nathanwilk7/adversarial-queries.git"
REPO = Path("/content/adversarial-queries")
PYTHON = Path("/content/advq-runtime-env/bin/python")
RUN_ID = datetime.now().strftime("%Y%m%d-%H%M%S")
DRIVE_RUN_ROOT = Path("/content/drive/MyDrive/adversarial-query-data/overnight-runs")
INIT_POINTS = 20
BO_CALLS = 20
QUERY_TIMEOUT_SECONDS = 30
SEED = 44
GRAMMAR_CHECK_SAMPLES = 20
MIN_GRAMMAR_VALID_FRACTION = 0.95
AUTO_DISCONNECT_WHEN_DONE = True
print({"run_id": RUN_ID, "init_points": INIT_POINTS, "bo_calls": BO_CALLS, "timeout_seconds": QUERY_TIMEOUT_SECONDS, "seed": SEED, "grammar_valid_fraction_required": MIN_GRAMMAR_VALID_FRACTION, "auto_disconnect": AUTO_DISCONNECT_WHEN_DONE})

## 1. Restore the repository and API secret

In [ ]:
import subprocess
if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(["git", "fetch", "origin", BRANCH], cwd=REPO, check=True)
    subprocess.run(["git", "checkout", BRANCH], cwd=REPO, check=True)
    subprocess.run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=REPO, check=True)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
print("Repository commit:", commit)

In [ ]:
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
assert os.environ.get("OPENAI_API_KEY"), "Add OPENAI_API_KEY to Colab Secrets first"
print("OPENAI_API_KEY is available (value hidden).")

## 2. Restore and verify the runtime

The idempotent bootstrap mounts Drive, restores DuckDB and trained models, starts vLLM/PostgreSQL/the worker, and runs queue, decoder, and real-oracle preflights.

In [ ]:
%cd /content/adversarial-queries
%run /content/adversarial-queries/genadversarialqueriesbo_resume.py

In [ ]:
import json
RUN_DIR = DRIVE_RUN_ROOT / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)
manifest = {"run_id": RUN_ID, "commit": commit, "branch": BRANCH, "init_points": INIT_POINTS, "bo_calls": BO_CALLS, "query_timeout_seconds": QUERY_TIMEOUT_SECONDS, "seed": SEED, "guided_decoding_backend": "outlines", "minimum_grammar_valid_fraction": MIN_GRAMMAR_VALID_FRACTION}
(RUN_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))
print("Durable run directory:", RUN_DIR)

## 3. Grammar-enforcement preflight

Sends saved embeddings through vLLM using GBNF and validates every result with the original Lark grammar. Invalid samples are retained for diagnosis. BO proceeds only if at least `MIN_GRAMMAR_VALID_FRACTION` pass; the optimization path independently rejects malformed decoder outputs before they reach the SQL oracle.

In [ ]:
grammar_code = f'''
import json
from pathlib import Path
from lark import Lark
import pyarrow.parquet as pq
from grammars import get_grammar
from optimization.query_inference.grammar_constrained_inference import GrammarConstrainedInference
data_path = Path("/content/drive/MyDrive/adversarial-query-data/cpu-artifacts/stack_embedded.parquet")
table = pq.read_table(data_path, columns=["embedding"]).slice(0, {GRAMMAR_CHECK_SAMPLES})
embeddings = [row["embedding"] for row in table.to_pylist()]
inference = GrammarConstrainedInference(grammar_name="stack", model_path="/content/advq-artifacts/stack-query-decoder", mapping_layer_path="/content/advq-artifacts/mapping_layer_epoch_0.pt", api_base_url="http://127.0.0.1:8000/v1", api_model_name="advq-decoder")
outputs = inference.generate_with_grammar_batch(embeddings, grammar=inference.grammar, max_tokens=128, temperature=0.7, max_concurrent=20)
parser = Lark(get_grammar("stack"), parser="lalr")
invalid = []
for index, output in enumerate(outputs):
    try: parser.parse(output)
    except Exception as error: invalid.append({{"index": index, "output": output, "error": str(error)}})
valid_count = len(outputs) - len(invalid)
valid_fraction = valid_count / len(outputs) if outputs else 0.0
result = {{"sample_count": len(outputs), "valid_count": valid_count, "valid_fraction": valid_fraction, "required_valid_fraction": {MIN_GRAMMAR_VALID_FRACTION}, "invalid": invalid, "outputs": outputs}}
Path({str(RUN_DIR / 'grammar-preflight.json')!r}).write_text(json.dumps(result, indent=2))
print(json.dumps({{k: v for k, v in result.items() if k != "outputs"}}, indent=2))
raise SystemExit(0 if len(outputs) == {GRAMMAR_CHECK_SAMPLES} and valid_fraction >= {MIN_GRAMMAR_VALID_FRACTION} else 2)
'''
grammar_log = RUN_DIR / "grammar-preflight.log"
environment = os.environ.copy()
environment.update({"PYTHONPATH": str(REPO), "MPLBACKEND": "Agg"})
with grammar_log.open("w") as handle:
    result = subprocess.run([str(PYTHON), "-u", "-c", grammar_code], cwd=REPO, env=environment, text=True, stdout=handle, stderr=subprocess.STDOUT)
PREFLIGHT_OK = result.returncode == 0
print(grammar_log.read_text())
print("Grammar preflight:", "PASS" if PREFLIGHT_OK else f"FAIL (exit {result.returncode})")

## 4. Bounded medium BO run

Runs in the foreground so Run all waits. Output is displayed and flushed to Drive. Nonzero exit codes are recorded so the morning report still runs.

In [ ]:
bo_log = RUN_DIR / "bo-console.log"
bo_status_path = RUN_DIR / "bo-status.json"
if not PREFLIGHT_OK:
    BO_RETURN_CODE = None
    bo_status_path.write_text(json.dumps({"status": "skipped", "reason": "grammar preflight failed"}, indent=2))
    print("BO SKIPPED because grammar enforcement failed.")
else:
    command = [str(PYTHON), "-u", str(REPO / "optimization/scripts/adversarial_query_optimization.py"), "--grammar_name", "stack", "--schema", "Stack", "--task_id", "adversarial_query_abs", "--db_backend", "duckdb", "--path_to_query_vae_model", "/content/advq-artifacts/stack-query-decoder", "--path_to_mapping_layer", "/content/advq-artifacts/mapping_layer_epoch_0.pt", "--path_to_plan_vae_statedict", "/content/advq-artifacts/best_64_stack.ckpt", "--api_base_url", "http://127.0.0.1:8000/v1", "--api_model_name", "advq-decoder", "--timeout_ms", str(QUERY_TIMEOUT_SECONDS * 1000), "--timeout_strategy", "constant", "--constant_timeout", str(QUERY_TIMEOUT_SECONDS), "--num_initialization_points", str(INIT_POINTS), "--max_n_oracle_calls", str(BO_CALLS), "--init_n_update_epochs", "10", "--num_update_epochs", "2", "--bsz", "1", "--save_freq", "2", "--seed", str(SEED), "--track_with_wandb", "False", "-", "run_lolbo", "-", "done"]
    environment = os.environ.copy()
    environment.update({"PYTHONPATH": str(REPO), "MPLBACKEND": "Agg", "WANDB_MODE": "disabled", "ORACLE_CONCURRENCY": "1", "ORACLE_INIT_BATCH_SIZE": "1"})
    with bo_log.open("w", buffering=1) as handle:
        process = subprocess.Popen(command, cwd=RUN_DIR, env=environment, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
        for line in process.stdout:
            print(line, end="")
            handle.write(line)
        BO_RETURN_CODE = process.wait()
    status = "complete" if BO_RETURN_CODE == 0 else "failed"
    bo_status_path.write_text(json.dumps({"status": status, "return_code": BO_RETURN_CODE}, indent=2))
    print(f"BO process: {status.upper()} (exit {BO_RETURN_CODE})")

## 5. Morning report

Inventories artifacts, summarizes statuses and advantages, and writes CSV/JSON reports to Drive.

In [ ]:
import pandas as pd
csv_paths = sorted(RUN_DIR.rglob("oracle_logs_abs/*.csv"), key=lambda path: path.stat().st_mtime)
frames = []
for path in csv_paths:
    try:
        frame = pd.read_csv(path)
        frame["source_file"] = str(path.relative_to(RUN_DIR))
        frames.append(frame)
    except Exception as error: print("Could not read", path, error)
all_results = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
if len(all_results):
    all_results.to_csv(RUN_DIR / "all-oracle-results.csv", index=False)
    complete = all_results[(all_results.default_status == "complete") & (all_results.generated_status == "complete")]
    summary = {"run_id": RUN_ID, "bo_return_code": BO_RETURN_CODE, "total_rows": len(all_results), "complete_pairs": len(complete), "timeout_pairs": int(((all_results.default_status == "timeout") | (all_results.generated_status == "timeout")).sum()), "error_pairs": int(((all_results.default_status == "error") | (all_results.generated_status == "error")).sum()), "positive_complete_pairs": int((complete.absolute_improvement_seconds > 0).sum()), "best_complete_advantage_seconds": float(complete.absolute_improvement_seconds.max()) if len(complete) else None}
    (RUN_DIR / "morning-summary.json").write_text(json.dumps(summary, indent=2))
    print(json.dumps(summary, indent=2))
    if len(complete):
        display(complete.sort_values("absolute_improvement_seconds", ascending=False)[["query", "generated_plan", "default_plan_time_seconds", "generated_plan_time_seconds", "absolute_improvement_seconds", "default_status", "generated_status"]].head(20))
else:
    summary = {"run_id": RUN_ID, "bo_return_code": BO_RETURN_CODE, "total_rows": 0}
    (RUN_DIR / "morning-summary.json").write_text(json.dumps(summary, indent=2))
    print("No oracle rows were produced. Inspect:", bo_log)
print("\nDurable artifacts:")
for path in sorted(RUN_DIR.rglob("*")):
    if path.is_file(): print(f"{path.stat().st_size:12,d}  {path.relative_to(RUN_DIR)}")
print("\nRun directory:", RUN_DIR)

## 6. Release the Colab runtime

After all durable results have been written to Drive, disconnects the runtime so an idle A100 is not left allocated. Set `AUTO_DISCONNECT_WHEN_DONE = False` in the first cell if you want to inspect the live runtime afterward.

In [ ]:
if AUTO_DISCONNECT_WHEN_DONE:
    import time
    print("All durable artifacts are saved in:", RUN_DIR)
    print("Releasing the Colab runtime in 10 seconds...")
    time.sleep(10)
    from google.colab import runtime
    runtime.unassign()
else:
    print("Runtime left connected because AUTO_DISCONNECT_WHEN_DONE=False")

## Tomorrow morning

Share `morning-summary.json`, `grammar-preflight.json`, and the tail of `bo-console.log`. Promising complete candidates must be rerun several times before calling them adversarial. Errors or mostly censored rows identify the stage to debug.